# Sensitivity analysis and operating-threshold selection

The final CDU checkpoint is evaluated on the fixed validation set of 200 images. The search space combines non-maximum suppression (NMS) IoU thresholds of 0.50, 0.60, 0.70, and 0.80 with confidence thresholds from 0.01 to 0.81. These IoU values control suppression among predicted boxes; TP, FP, and FN come from the evaluator confusion matrix. The file `run_model/results/sensitivity_results.csv` contains the resulting 972 observations. mAP@50 remains the quality reference, while the operating point is chosen by maximizing the F1 score independently for each class, balancing precision and recall.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import Image, display

START = Path.cwd().resolve()
CANDIDATE_ROOTS = (START, *START.parents, START / 'repo')
ROOT = next((path for path in CANDIDATE_ROOTS
             if (path / 'run.py').is_file()
             and (path / 'active_learning_benchmark').is_dir()), None)
if ROOT is None:
    raise RuntimeError('The repository root could not be located.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from active_learning_benchmark.paper_plots import sensitivity_figures
from active_learning_benchmark.sensitivity_visuals import (
    build_analysis_specs, build_heatmap_figure,
)

CSV = ROOT / 'run_model/results/sensitivity_results.csv'
FIGURES = ROOT / 'figures/paper'
data = pd.read_csv(CSV)
artifacts = sensitivity_figures(ROOT / 'run_model')
analysis_specs, metric_best_df, f1_selection_df = build_analysis_specs(data)


## From box matching to evaluation metrics

![Relationship among the confusion matrix, IoU, precision, and recall](../figures/sensitivity/metrics.jpg)

*Intersection over Union* (IoU) measures the overlap between two bounding boxes, as defined in the paper:

$$
\mathrm{IoU}(B_{\mathrm{pred}},B_{\mathrm{gt}}) = \frac{|B_{\mathrm{pred}} \cap B_{\mathrm{gt}}|}{|B_{\mathrm{pred}} \cup B_{\mathrm{gt}}|}.
$$

The same geometric measure appears in two distinct stages. During evaluation, $B_{\mathrm{pred}}$ is a predicted box and $B_{\mathrm{gt}}$ is its ground-truth reference; a valid same-class match contributes a true positive (TP). An unmatched prediction contributes a false positive (FP), while an unrecovered annotated object contributes a false negative (FN). During NMS, IoU is instead calculated between competing predicted boxes, and its threshold controls the removal of redundant detections. Precision, $TP/(TP+FP)$, measures detection reliability; recall, $TP/(TP+FN)$, measures how many annotated defects were recovered.


## Interpreting an IoU threshold

![Examples of TP, FP, FN, and TN with an illustrative IoU threshold of 0.50](../figures/sensitivity/iou_example.jpg)

The figure uses 0.50 only as a didactic example: an approximate overlap of 0.70 is accepted as a TP, whereas 0.30 does not establish a match. From the prediction perspective, the latter contributes an FP; the unmatched annotation contributes an FN. The panels separate these two error perspectives. True negatives (TN) are useful conceptually but are not accumulated directly in object detection because the background is not a finite set of candidate boxes.

In the published sensitivity space, the `IoU` column stores the NMS threshold; the didactic value in the figure does not define this experimental parameter. Changing NMS alters which redundant predicted boxes remain after post-processing and can consequently change TP, FP, FN, precision, recall, and F1.


## Curves by class and NMS IoU

Each panel overlays the twelve curves formed by three classes and four NMS IoU thresholds, retaining one color per class and one line style per threshold. For `crack`, maximizing mAP@50 alone favors a conservative confidence threshold and recovers few defects. The F1 maximum shifts the decision toward a lower confidence threshold, which is appropriate when missing a real defect is more costly than accepting a moderate increase in false positives.


In [ ]:
display(Image(filename=str(FIGURES / 'sensitivity_legend.png')))

for filename in ('sensitivity_cl_ap50.png', 'sensitivity_cl_p.png',
                 'sensitivity_f1_score.png', 'sensitivity_TPR.png'):
    display(Image(filename=str(FIGURES / filename)))


## Critical case: `crack` at NMS IoU 0.70

Confidence 0.23 produces the highest class mAP@50 at this NMS threshold but recovers only one object. Confidence 0.08 maximizes F1 and retains greater sensitivity. In the table, TP represents correctly detected `crack` objects, FP represents detections without a matching annotation, and FN represents annotated objects not recovered by the model. These counts are obtained from the evaluator confusion matrix after NMS post-processing.


In [ ]:
crack_thresholds = artifacts['crack_thresholds'][
    ['confidence', 'tp', 'fp', 'fn', 'cl_f1', 'cl_ap50', 'cl_p', 'cl_r']
].sort_values('confidence', ascending=False).rename(columns={
    'confidence': 'Confidence', 'tp': 'TP', 'fp': 'FP', 'fn': 'FN',
    'cl_f1': 'F1', 'cl_ap50': 'AP50',
    'cl_p': 'Precision', 'cl_r': 'Recall',
})
display(crack_thresholds.style.format({
    'Confidence': '{:.2f}', 'F1': '{:.6f}', 'AP50': '{:.6f}',
    'Precision': '{:.6f}', 'Recall': '{:.6f}',
}))


## Metric optima and final selection

The first table reports the best observed NMS IoU-confidence combination for each metric. The second applies the operating rule adopted in the study: maximum F1 independently for every class and for the overall result. The selected settings are `crack` 0.70/0.08, `patch` 0.50/0.20, and `pothole` 0.60/0.10.


In [ ]:
metric_best_display = metric_best_df.rename(columns={
    'graph': 'Graph', 'metric': 'Metric', 'goal': 'Goal',
    'optimal_iou': 'Optimal NMS IoU',
    'optimal_confidence': 'Optimal confidence',
    'optimal_metric_value': 'Optimal metric value',
    'score_at_optimal': 'Score at optimum',
})
f1_selection_display = f1_selection_df.rename(columns={
    'graph': 'Graph', 'selected_by': 'Selected by',
    'iou': 'NMS IoU', 'confidence': 'Confidence', 'f1_value': 'F1',
    'ap50_at_f1': 'AP50 at maximum F1',
    'precision_at_f1': 'Precision at maximum F1',
    'recall_at_f1': 'Recall at maximum F1',
    'tp_at_f1': 'TP at maximum F1', 'fp_at_f1': 'FP at maximum F1',
    'fn_at_f1': 'FN at maximum F1',
})
print('Best operating point for each metric, by class and overall:')
display(metric_best_display)
print('Final F1-based selection, by class and overall:')
display(f1_selection_display.sort_values('Graph'))


## Heatmaps over the NMS IoU-confidence space

F1 was selected because its harmonic mean summarizes the balance between precision and recall and penalizes settings in which one is high at the expense of the other. Precision alone would favor overly restrictive thresholds and could increase missed defects; recall alone would favor permissive thresholds and increase false positives. mAP@50 remains the global quality reference, but its isolated maximum does not necessarily define the best operating point: for `crack`, for example, it occurs at confidence 0.23 and recovers only one object. TP, FP, and FN clarify this behavior, but their raw counts depend on class frequency and do not independently provide a balanced measure.

The heatmaps therefore show the F1 surface over the NMS IoU-confidence space and mark each class maximum with a circle, without introducing an additional aggregate score. For `crack`, 0.70/0.08 prioritizes coverage because of the cost of missing real defects; `patch` has a broader favorable region around 0.50/0.20; and `pothole` reaches its balance at 0.60/0.10. Independent class-wise selection avoids imposing one threshold pair on categories with different frequencies and detection difficulty.


In [ ]:
print('Two-dimensional F1 heatmaps:')
build_heatmap_figure(analysis_specs, 'F1')
display(Image(filename=str(FIGURES / 'heatmap_2d_f1.png')))
